# TunedAI — 4-Model Causal Reasoning Comparison

**Compare four models on the same causal reasoning questions:**
- Base Qwen 2.5-7B (untuned)
- GPT-4o (OpenAI)
- Claude 3.5 Sonnet (Anthropic)
- **TunedAI Causal Model** (Qwen 2.5-7B fine-tuned on causal reasoning)

All test questions come from **pre-AI sources** — classic epidemiology, statistics, and philosophy texts published decades before large language models existed. Correct answers were established by human experts.

### Sources:
- John Snow (1854) — *On the Mode of Communication of Cholera*
- E.H. Simpson (1951) — *The Interpretation of Interaction in Contingency Tables*
- Bradford Hill (1965) — *The Environment and Disease: Association or Causation?*
- David Hume (1748) — *An Enquiry Concerning Human Understanding*
- R.A. Fisher (1935) — *The Design of Experiments*
- Ignaz Semmelweis (1847) — Vienna General Hospital records

---
**Setup:** Select `Runtime > Change runtime type > T4 GPU`

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch openai anthropic

In [ ]:
# ── API KEYS (optional — only needed for GPT-4 and Claude comparisons) ──
# Leave blank to skip that model
OPENAI_API_KEY   = ""   # sk-...
ANTHROPIC_API_KEY = ""  # sk-ant-...

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedai/causal-reasoning-qwen-7b"  # HuggingFace public repo

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading base model (this takes ~2 min on T4)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Loading TunedAI causal adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()

print("✓ Local models ready")

In [ ]:
import openai, anthropic

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

SYSTEM = "You are a careful reasoner. Answer questions about causation, association, intervention, and counterfactuals precisely and correctly."

def ask_local(question, use_adapter=True, max_new_tokens=350):
    """Query base Qwen or TunedAI model."""
    if use_adapter:
        tuned_model.enable_adapter_layers()
    else:
        tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(tuned_model.device)
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(question):
    """Query GPT-4o."""
    if not oai_client:
        return "[No OpenAI API key provided]"
    r = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":question}],
        max_tokens=400
    )
    return r.choices[0].message.content.strip()

def ask_claude(question):
    """Query Claude 3.5 Sonnet."""
    if not ant_client:
        return "[No Anthropic API key provided]"
    r = ant_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role":"user","content":question}]
    )
    return r.content[0].text.strip()

def compare_all(question, source="", rung=""):
    """Run all four models on the same question and display side-by-side."""
    SEP = "=" * 72
    DIV = "-" * 72
    print(SEP)
    if source: print(f"SOURCE : {source}")
    if rung:   print(f"RUNG   : {rung}")
    print(SEP)
    print(f"QUESTION:\n{question}\n")

    models = [
        ("BASE QWEN 2.5-7B (untuned)",  lambda q: ask_local(q, use_adapter=False)),
        ("GPT-4o",                        ask_gpt4),
        ("CLAUDE 3.5 SONNET",             ask_claude),
        ("TUNEDAI CAUSAL MODEL ★",        lambda q: ask_local(q, use_adapter=True)),
    ]
    for name, fn in models:
        print(DIV)
        print(f"[ {name} ]")
        print(DIV)
        print(fn(question))
        print()

print("✓ Comparison functions ready")

---
## Test 1 — Simpson's Paradox (E.H. Simpson, 1951)

**Source:** *The Interpretation of Interaction in Contingency Tables*, E.H. Simpson, 1951

A treatment that is better in every subgroup can appear worse in the aggregate. Resolving it requires causal reasoning about confounders — pure statistics cannot answer which treatment to give.

**Rung 2 — Intervention / Confounding**

In [ ]:
compare_all(
    question="""Kidney stone treatment study:

Small stones: Treatment A 93% success (81/87), Treatment B 87% success (234/270)
Large stones:  Treatment A 73% success (192/263), Treatment B 69% success (55/80)
Combined:      Treatment A 78% (273/350), Treatment B 83% (289/350)

Treatment A is better for small stones AND large stones individually, yet appears worse overall.
A patient arrives and we do not yet know their stone size.
Which treatment should we administer, and why is the combined percentage wrong?""",
    source="E.H. Simpson — The Interpretation of Interaction in Contingency Tables (1951)",
    rung="Rung 2 (Intervention) — Confounding"
)

---
## Test 2 — John Snow's Cholera Investigation (1854)

**Source:** *On the Mode of Communication of Cholera*, John Snow, 1854

Snow compared death rates between households served by two water companies, then removed the Broad Street pump handle. The two steps represent Rung 1 and Rung 2 reasoning.

**Rung 1 → Rung 2**

In [ ]:
compare_all(
    question="""In 1854, John Snow observed:
- Southwark & Vauxhall water customers: 315 cholera deaths per 10,000 houses
- Lambeth water customers: 37 deaths per 10,000 houses

He then removed the handle from the Broad Street pump. The local outbreak subsided.

What does the household comparison between water companies establish?
What does the pump handle removal establish that the comparison alone cannot?
Why is the direction of Snow's reasoning — from mechanism hypothesis to targeted intervention — important?""",
    source="John Snow — On the Mode of Communication of Cholera (1854)",
    rung="Rung 1 (Association) → Rung 2 (Intervention)"
)

---
## Test 3 — Semmelweis Handwashing (1847)

**Source:** Vienna General Hospital records, Ignaz Semmelweis, 1847

Semmelweis observed a mortality gap between two wards, then intervened with mandatory handwashing.

**Rung 2 — Intervention**

In [ ]:
compare_all(
    question="""In 1847, Semmelweis observed:
- Doctors' ward (performed autopsies before deliveries): 10-35% childbed fever mortality
- Midwives' ward (no autopsies): 1-2% mortality

He mandated chlorinated lime handwashing before entering the maternity ward.
The doctors' ward mortality immediately fell to under 2%.

Before the intervention, could Semmelweis conclude that autopsy exposure caused the deaths?
After the intervention, what can he conclude that he could not before?
Explain the difference between what the observation tells us and what the intervention tells us.""",
    source="Ignaz Semmelweis — Vienna General Hospital Records (1847)",
    rung="Rung 1 → Rung 2 (Intervention)"
)

---
## Test 4 — Bradford Hill Criteria (1965)

**Source:** *The Environment and Disease: Association or Causation?*, Bradford Hill, 1965

Can we infer causation without an RCT? Bradford Hill proposed 9 criteria.

**Rung 1 — Criteria for Causal Inference**

In [ ]:
compare_all(
    question="""Bradford Hill (1965) on smoking and lung cancer — no RCT was conducted:

- Smokers have 9-10x higher lung cancer rates than non-smokers (strength)
- Finding replicates across countries, decades, and study designs (consistency)
- Lung cancer specifically, not all cancers equally (specificity)
- Cancer diagnosis comes after years of smoking, not before (temporality)
- Heavier smokers have higher rates than lighter smokers (dose-response gradient)
- Carcinogens in cigarette smoke are biologically plausible causes (plausibility)

Critics: without an RCT, this is only association. Can we conclude smoking causes lung cancer?
What is the strongest argument for and against causal inference without an intervention?""",
    source="Bradford Hill — The Environment and Disease: Association or Causation? (1965)",
    rung="Rung 1 — Criteria for Causal Inference"
)

---
## Test 5 — Hume's Billiard Balls and Overdetermination (1748)

**Source:** David Hume — *An Enquiry Concerning Human Understanding* (1748)

Hume's counterfactual definition of causation succeeds in simple cases but fails for overdetermination — a failure that motivated 250 years of philosophy and eventually Pearl's structural causal models.

**Rung 3 — Counterfactual**

In [ ]:
compare_all(
    question="""Hume (1748): 'We may define a cause to be an object where, if the first object had not been, the second never had existed.'

Apply this but-for test to two cases:

Case 1: A driver runs a red light and strikes a pedestrian who is injured.
But-for test: Would the injury have occurred if the driver had stopped? No. Test passes.

Case 2 (overdetermination): Two assassins independently poison a victim's drink.
Assassin A adds poison. Assassin B (not knowing this) also adds a lethal dose. The victim dies.
But-for Assassin A: victim still dies (B's poison kills). But-for Assassin B: victim still dies (A's poison kills).

Does Hume's but-for test correctly identify the cause in Case 2? If not, what does this reveal about
the limits of counterfactual theories of causation?""",
    source="David Hume — An Enquiry Concerning Human Understanding (1748)",
    rung="Rung 3 (Counterfactual) — Overdetermination"
)

---
## Test 6 — Fisher's Randomization (1935)

**Source:** R.A. Fisher — *The Design of Experiments* (1935)

Fisher introduced randomization as the mechanism that licenses causal inference. The same drug can appear harmful in observational data and beneficial in an RCT.

**Rung 2 — Why randomization works**

In [ ]:
compare_all(
    question="""Two studies of the same drug:

Study A (observational): Doctors prescribe the drug to sicker patients.
Result: Drug group has WORSE outcomes than no-drug group.

Study B (randomized, Fisher 1935): Patients randomly assigned to drug or no drug.
Result: Drug group has BETTER outcomes than no-drug group.

Why do these studies give opposite results?
What does randomization do mathematically that observational assignment cannot?
Why does Study B justify a causal conclusion and Study A does not — even if Study A has 10x more patients?""",
    source="R.A. Fisher — The Design of Experiments (1935)",
    rung="Rung 2 (Intervention) — Randomization"
)

---
## Your Own Question

In [ ]:
YOUR_QUESTION = """
Type your causal reasoning question here.
"""

compare_all(YOUR_QUESTION, source="Custom")

---
## About TunedAI

We fine-tune open-source LLMs for real-world reasoning tasks.

Our causal reasoning model scores **96.96%** on the CLadder benchmark — within 1.6% of the theoretical ceiling. General-purpose LLMs score in the low 60s on the same test.

**Want this capability for your domain?** → [tunedai.ai](https://tunedai.ai)